# Business Mission 1: High-Value Customer Segmentation & Tiering

## 1. Stakeholder & Context
* **Stakeholder:** Chief Marketing Officer (CMO)
* **Background:** The marketing team is currently allocating promotional budget ($50 vouchers) uniformly across all customers in São Paulo (`SP`), regardless of historical purchasing behavior.

## 2. Business Problem & Risk
* **Problem:** Broad, untargeted discounting creates a high Customer Acquisition Cost (CAC) and wastes marketing capital on one-time or low-value buyers.
* **Risk:** Inefficient budget spend and missed revenue opportunities from unrewarded high-value customers.

## 3. Core Business Question
> *"Can we isolate all São Paulo customers, aggregate their total lifetime spend, and categorize them into strict marketing tiers so we can reserve high-value promotional vouchers exclusively for VIP buyers?"*

## 4. Expected Action / Deliverable
Deliver a categorized customer list identifying:
* **VIP Customer:** Lifetime Spend $\ge \$1,000$
* **Regular Customer:** Lifetime Spend between $\$200$ and $\$999.99$
* **Low Value Customer:** Lifetime Spend $< \$200$

The marketing team will export this target list directly into their campaign engine.


In [ ]:
from db_utils import run_pg_query

cmo_segmentation_query = """
SELECT 
    c.customer_id,
    c.customer_state,
    SUM(p.payment_value) AS lifetime_spend,
    CASE 
        WHEN SUM(p.payment_value) >= 1000 THEN 'VIP Customer'
        WHEN SUM(p.payment_value) BETWEEN 200 AND 999.99 THEN 'Regular Customer'
        ELSE 'Low Value Customer'
    END AS marketing_tier
FROM 
    olist_ecommerce.customers c
INNER JOIN 
    olist_ecommerce.orders o ON c.customer_id = o.customer_id
INNER JOIN 
    olist_ecommerce.order_payments p ON o.order_id = p.order_id
WHERE 
    c.customer_state = 'SP'
GROUP BY 
    c.customer_id, c.customer_state
ORDER BY 
    lifetime_spend DESC
LIMIT 10;
"""

df_cmo_tiers = run_pg_query(cmo_segmentation_query)
df_cmo_tiers


In [ ]:
from db_utils import run_mysql_query

query = """
        SELECT * 
        FROM avocado.avoca
        LIMIT 100
        """
result = run_mysql_query(query)
result.head()

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
load_dotenv(override=True)
mysql_url = os.getenv("MYSQL_URL")
mysql_engine = create_engine(
    mysql_url,
    pool_pre_ping=True
)
query = """
        SELECT * 
        FROM avocado.avoca 
        LIMIT 100;
        """
df = pd.read_sql(query, con = mysql_engine)
df